# 01 — Ingest MLB GUMBO Feed

Pulls game schedule + play-by-play GUMBO JSON from `statsapi.mlb.com` into the
UC Volume landing zone. Notebook 02 picks these files up with Auto Loader.

## Why this matters — Well-Architected Framework

| WAF pillar | What this notebook does, and why it's good |
|------------|---------------------------------------------|
| **Operational Excellence** | Season/team/date window is parameterized via `.env` — the same notebook code runs as a **one-shot demo**, a **daily backfill job**, or a **team-scoped test**. No ifs, no branching, no forks. |
| **Reliability** | Files already present are skipped, so re-runs are idempotent. If the notebook dies halfway through, re-running finishes the job cleanly. |
| **Cost Optimization** | Targeted date/team windows mean we download *exactly* what the demo needs — a full season is ~2,430 games (~2 GB); a single team is ~162. |

> **Tip for MLB demo:** set `MLB_TEAM_ID` in `.env` to the team you're
> presenting to. The whole pipeline still runs, just against 162 games instead of 2,430 —
> the story is identical, the runtime is an order of magnitude faster.


In [1]:
from databricks.connect import DatabricksSession
from databricks.sdk import WorkspaceClient
from dotenv import load_dotenv
import os, json, io, time, requests
from datetime import datetime

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()
w = WorkspaceClient()

UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA",  "mlb_gumbo_waf")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/raw_gumbo"

SEASON     = int(os.getenv("MLB_SEASON", "2025"))
SPORT_ID   = int(os.getenv("MLB_SPORT_ID", "1"))
TEAM_ID    = os.getenv("MLB_TEAM_ID", "").strip() or None
START_DATE = os.getenv("MLB_START_DATE", "").strip() or f"{SEASON}-03-01"
END_DATE   = os.getenv("MLB_END_DATE", "").strip() or f"{SEASON}-11-05"

print(f"Volume landing zone: {VOLUME_PATH}")
print(f"Season: {SEASON}, sportId: {SPORT_ID}, team: {TEAM_ID or 'ALL'}, window: {START_DATE} -> {END_DATE}")


Volume landing zone: /Volumes/alexander_booth/mlb_gumbo_waf_bronze/raw_gumbo
Season: 2025, sportId: 1, team: ALL, window: 2025-03-01 -> 2025-11-01


## Fetch the schedule → list of gamePks

> **WAF — Reliability.** We filter to games whose `abstractGameState` is
> `Final` or `Live` — postponed/cancelled games never have a usable GUMBO
> payload, and fetching them just produces 404s and noise in our logs.

In [2]:
team_qs = f"&teamId={TEAM_ID}" if TEAM_ID else ""
schedule_url = (
    f"https://statsapi.mlb.com/api/v1/schedule?"
    f"sportId={SPORT_ID}&gameType=R,D,F,L,W"
    f"&startDate={START_DATE}&endDate={END_DATE}{team_qs}"
)
print(schedule_url)

schedule = requests.get(schedule_url, timeout=60).json()

game_pks = []
for d in schedule.get("dates", []):
    for g in d.get("games", []):
        if g.get("status", {}).get("abstractGameState") in ("Final", "Live"):
            game_pks.append(g["gamePk"])

game_pks = sorted(set(game_pks))
print(f"Found {len(game_pks):,} games to fetch.")


https://statsapi.mlb.com/api/v1/schedule?sportId=1&gameType=R,D,F,L,W&startDate=2025-03-01&endDate=2025-11-01


Found 2,477 games to fetch.


## Download each game's GUMBO feed into the UC Volume

Files land at `raw_gumbo/year=YYYY/month=MM/day=DD/game_data_{gamePk}.json`.

> **WAF — Operational Excellence.** The `if os.path.exists(target): skip`
> check is what turns this from a fragile script into an idempotent batch
> job. You can schedule this notebook every 15 minutes and it will only
> download games that finished since the last run.

In [3]:
today = datetime.utcnow()
partition = f"year={today.year}/month={today.month:02d}/day={today.day:02d}"
out_dir = f"{VOLUME_PATH}/{partition}"

# Pre-list existing files in the target partition so we can skip cheaply.
existing = set()
try:
    for f in w.files.list_directory_contents(out_dir):
        existing.add(f.name)
except Exception:
    # Directory doesn't exist yet — that's fine, first run.
    pass
print(f"Already in {out_dir}: {len(existing)} files")

downloaded = 0
skipped    = 0
failed     = []

for i, gpk in enumerate(game_pks, 1):
    name = f"game_data_{gpk}.json"
    if name in existing:
        skipped += 1
        continue
    url = f"https://statsapi.mlb.com/api/v1.1/game/{gpk}/feed/live?hydrate=credits,alignment,flags,officials,preState"
    try:
        r = requests.get(url, timeout=90)
        r.raise_for_status()
        payload = r.content  # bytes
        w.files.upload(f"{out_dir}/{name}", payload, overwrite=True)
        downloaded += 1
    except Exception as e:
        failed.append((gpk, str(e)[:80]))
    if i % 50 == 0:
        print(f"  {i:>5} / {len(game_pks):,}  (downloaded={downloaded}, skipped={skipped}, failed={len(failed)})")

print(f"\nDone. downloaded={downloaded}, skipped={skipped}, failed={len(failed)}")
if failed[:5]:
    print("  First failures:", failed[:5])


/var/folders/xn/_5vq53hj5ld7v66jgsmc64f40000gp/T/ipykernel_60288/3798606974.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


Already in /Volumes/alexander_booth/mlb_gumbo_waf_bronze/raw_gumbo/year=2026/month=04/day=21: 179 files


     50 / 2,477  (downloaded=50, skipped=0, failed=0)


    100 / 2,477  (downloaded=100, skipped=0, failed=0)


    150 / 2,477  (downloaded=150, skipped=0, failed=0)


...

   2100 / 2,477  (downloaded=2100, skipped=0, failed=0)


   2150 / 2,477  (downloaded=2150, skipped=0, failed=0)


   2400 / 2,477  (downloaded=2221, skipped=179, failed=0)


   2450 / 2,477  (downloaded=2271, skipped=179, failed=0)



Done. downloaded=2298, skipped=179, failed=0


In [4]:
try:
    files = [f for f in w.files.list_directory_contents(out_dir) if f.name.endswith('.json')]
    print(f"Landed {len(files)} files into {out_dir}")
    if files:
        sample = files[0]
        size_mb = (sample.file_size or 0) / 1e6
        print(f"  Sample: {sample.name} ({size_mb:.2f} MB)")
except Exception as e:
    print(f"Could not list: {e}")


Landed 2477 files into /Volumes/alexander_booth/mlb_gumbo_waf_bronze/raw_gumbo/year=2026/month=04/day=21
  Sample: game_data_776135.json (1.32 MB)
